# 10k Where does the personal map stop paying? (NSD budget curve)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 10a and 10b.

Development analysis, not a prespecified test. The 8 NSD people, DINOv2 features, OLS and ridge donors. Labelled budgets of 500, 1,000, 2,000, 4,000 images and
the whole pool: borrowing, the low-rank map, own ridge on the same images, and the combination; encoding r and
identification among 200 (raw and centred).

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '10k-budgetcurve', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"K": 200, "folds": 5, "n_galleries": 4, "gallery_size": 200, "gallery_seed": 346, "delta_r_share": 0.05, "roi_classes": ["prf-visualrois", "floc-faces", "floc-places", "floc-bodies", "floc-words"], "ks": [500, 1000, 2000, 4000, "all"], "draw": 13, "ridge_seed": 13, "feature": "dinov2", "wall_hours": 11.0}')
SUBJECTS = ['subj01', 'subj02', 'subj03', 'subj04', 'subj05', 'subj06', 'subj07', 'subj08']

import numpy as np
import pandas as pd
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

if not isinstance(FIXED["K"], int):
    raise RuntimeError("K is not fixed: fill it in from 09q before the confirmation runs")
IN = Path("/kaggle/input")
conv = next(p.parent for p in IN.rglob("shared_train_nsd_ids.npy"))
shared = np.load(conv / "shared_train_nsd_ids.npy")
cols, common = {}, {}
for h in ("lh", "rh"):
    masks = [np.load(next((conv / f"{s}_roi").rglob(f"{h}.all-vertices_fsaverage_space.npy"))) for s in SUBJECTS]
    common[h], cl = nc.common_vertices(masks)
    cols[h] = dict(zip(SUBJECTS, cl))
V = sum(len(common[h]) for h in common)
ids, Y = {}, {}
for s in SUBJECTS:
    ids[s] = np.load(conv / f"{s}_train_nsd_ids.npy")
    Y[s] = np.concatenate([np.load(conv / f"{s}_{h}_fmri.npy", mmap_mode="r")[:, cols[h][s]] for h in ("lh", "rh")], 1)
noise = np.concatenate([np.concatenate([np.load(conv / f"{s}_{h}_noise_ceiling.npy").astype(np.float64)[cols[h][s]] for h in ("lh", "rh")]) for s in SUBJECTS])
DELTA_R = FIXED["delta_r_share"] * float(np.median(np.sqrt(np.clip(noise, 0, None) / 100)))
FEATS = {}
for name in ("alexnet", "dinov2"):
    f = np.load(next(IN.rglob(f"10b_{name}_kpca1000.npz")))
    FEATS[name] = (f["z"].astype(np.float64), {int(i): j for j, i in enumerate(f["nsd_ids"])})
feats = lambda name, nsd_ids: FEATS[name][0][[FEATS[name][1][int(i)] for i in nsd_ids]]
test_ids = np.sort(shared)
rows_of = {s: {int(i): j for j, i in enumerate(ids[s])} for s in SUBJECTS}
test_rows = {s: np.array([rows_of[s][int(i)] for i in test_ids]) for s in SUBJECTS}
pool_rows = {s: np.setdiff1d(np.arange(len(ids[s])), test_rows[s]) for s in SUBJECTS}
GALS = nc.galleries(len(test_ids), FIXED["n_galleries"], FIXED["gallery_size"], FIXED["gallery_seed"])
rois = {}
for cls in FIXED["roi_classes"]:
    for s in SUBJECTS:
        lab = np.concatenate([np.load(next((conv / f"{s}_roi").rglob(f"{h}.{cls}_challenge_space.npy")))[cols[h][s]] for h in ("lh", "rh")])
        rois[(s, cls)] = lab
print(f"{V} common vertices; delta_r {DELTA_R:.4f}; {len(test_ids)} test images; pools", {s: len(pool_rows[s]) for s in SUBJECTS},
      "|", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
def ols_coef(x, y):
    xa = np.concatenate([np.ones((len(x), 1)), x], 1)
    return np.linalg.solve(xa.T @ xa, xa.T @ np.asarray(y, np.float64)).astype(np.float32)
def ridge_coef(x, y):
    m = RidgePerOutput(lm.ALPHAS, FIXED["folds"], FIXED["ridge_seed"]).fit(x, np.asarray(y, np.float32))
    return np.vstack([(m.y_mean_ - m.x_mean_ @ m.coef_)[None], m.coef_]).astype(np.float32)
lin_pred = lambda beta, x: np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64)
F = FIXED["feature"]
BETA = {}
for kind, fit in (("ols", ols_coef), ("ridge", ridge_coef)):
    for s in SUBJECTS:
        BETA[(kind, s)] = fit(feats(F, ids[s][pool_rows[s]]), Y[s][pool_rows[s]])
        print(kind, s, "donor encoder |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
# donors fitted: from here only the current person's responses stay in memory
del Y
load_y = lambda s: np.concatenate([np.load(conv / f"{s}_{h}_fmri.npy", mmap_mode="r")[:, cols[h][s]] for h in ("lh", "rh")], 1)

In [ ]:
def oof(fit_predict, k, seed):
    split = np.array_split(np.random.default_rng(seed).permutation(k), FIXED["folds"])
    out = None
    for held in split:
        pred = fit_predict(np.setdiff1d(np.arange(k), held), held)
        if out is None:
            out = np.empty((k, pred.shape[1]), np.float32)
        out[held] = pred
    return out

rows, stopped = [], None
d = FIXED["draw"]
for p, s in enumerate(SUBJECTS, start=1):
    t = time.monotonic()
    donors = [x for x in SUBJECTS if x != s]
    y_s = load_y(s)
    meas = np.asarray(y_s[test_rows[s]], np.float64)
    mc = meas - meas.mean(0)
    y_pool = np.asarray(y_s[pool_rows[s]], np.float32)
    del y_s
    x_pool, x_test = feats(F, ids[s][pool_rows[s]]), feats(F, test_ids)
    def record(method, donors_kind, k, pred, y_mean=None, pred_k_mean=None):
        pc = pred - pred.mean(0)
        rv = (pc * mc).sum(0) / np.sqrt((pc ** 2).sum(0) * (mc ** 2).sum(0) + 1e-30)
        # raw scores: the prediction's mean over the labelled images replaced by the person's
        raw = pred if y_mean is None else pred - pred_k_mean + y_mean
        rows.append({"subject": s, "method": method, "donors": donors_kind, "k": k, "enc_r": float(rv.mean()),
                     "top1_raw": float(np.mean([nc.identify_top1(raw[g], meas[g]) for g in GALS])),
                     "top1_centred": float(np.mean([nc.identify_top1(pred[g], meas[g], centred=True) for g in GALS]))})
    order = np.random.default_rng([d, p]).permutation(len(pool_rows[s]))
    avg = {}
    for kind in ("ols", "ridge"):
        acc_p, acc_t = np.zeros((len(x_pool), V), np.float32), np.zeros((len(x_test), V), np.float32)
        for dn in donors:
            acc_p += lin_pred(BETA[(kind, dn)], x_pool).astype(np.float32)
            acc_t += lin_pred(BETA[(kind, dn)], x_test).astype(np.float32)
        avg[kind] = (acc_p / len(donors), (acc_t / len(donors)).astype(np.float64))
        record("borrowing", kind, 0, avg[kind][1])
        del acc_p, acc_t
    basis = {kind: lm.prediction_basis(avg[kind][0], FIXED["K"]) for kind in avg}
    for k in FIXED["ks"]:
        idx = order if k == "all" else order[:k]
        kk = len(idx)
        y_k = y_pool[idx].astype(np.float64)
        xk = x_pool[idx]
        own = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(xk, y_k)
        own_t, y_mean = own.predict(x_test), y_k.mean(0)
        own_oof = oof(lambda tr, he: own.refit(xk[tr], y_k[tr]).predict(xk[he]), kk, d)
        record("own_ridge", "-", k, own_t, y_mean, own.predict(xk).mean(0))
        for kind in ("ols", "ridge"):
            mean, B = basis[kind]
            a_k, a_t = avg[kind][0][idx].astype(np.float64), avg[kind][1]
            model = lm.fit_lowrank_map(a_k, y_k, mean, B, seed=d)
            m_t = lm.apply_lowrank_map(model, a_t, mean, B)
            record("map", kind, k, m_t, y_mean, lm.apply_lowrank_map(model, a_k, mean, B).mean(0))
            m_oof = oof(lambda tr, he: lm.apply_lowrank_map(lm.fit_lowrank_map(a_k[tr], y_k[tr], mean, B, seed=d), a_k[he], mean, B), kk, d)
            w = nc.stack_fit([m_oof, own_oof], y_k)
            record("combination", kind, k, nc.stack_apply(w, [m_oof, own_oof], [m_t, own_t], y_mean))
            del a_k, m_oof
        pd.DataFrame(rows).to_csv(C.METRICS_DIR / "10k_scores.csv", index=False)
        print(s, "k", k, f"({kk})", {r["method"] + "|" + r["donors"]: round(r["top1_centred"], 1) for r in rows[-5:]},
              f"| {(time.monotonic() - T0) / 60:.1f} min", flush=True)
        del own, own_oof, xk, y_k
        if (time.monotonic() - T0) / 3600 > FIXED["wall_hours"]:
            stopped = f"wall clock at {s} k {k}"
            break
    del avg
    if stopped:
        print("STOPPED:", stopped)
        break

In [ ]:
frame = pd.DataFrame(rows)
res = {"post_hoc": "development: NSD people of P-7; the map's budget curve to the whole recording", "means": {}, "contrasts": {}}
for (method, dk, k), g in frame.groupby(["method", "donors", "k"], sort=False):
    res["means"][f"{method}|{dk}|{k}"] = {m: float(g[m].mean()) for m in ("enc_r", "top1_raw", "top1_centred")} | {"n": int(len(g))}
def diff(a, b, k, m):
    x = frame[(frame.method == a[0]) & (frame.donors == a[1]) & (frame.k.astype(str) == str(k))].set_index("subject")[m]
    y = frame[(frame.method == b[0]) & (frame.donors == b[1]) & (frame.k.astype(str) == str(k))].set_index("subject")[m]
    z = (x - y).dropna()
    return {"mean": float(z.mean()), "positive": int((z > 0).sum()), "n": int(len(z))}
for k in FIXED["ks"]:
    for m in ("enc_r", "top1_centred", "top1_raw"):
        for kind in ("ols", "ridge"):
            res["contrasts"][f"map_{kind}_minus_own|{k}|{m}"] = diff(("map", kind), ("own_ridge", "-"), k, m)
            res["contrasts"][f"combination_{kind}_minus_own|{k}|{m}"] = diff(("combination", kind), ("own_ridge", "-"), k, m)
res["stopped"] = stopped
(C.METRICS_DIR / "10k_summary.json").write_text(json.dumps(res, indent=2))
print("10K_SUMMARY_JSON " + json.dumps(res))